# Satellite Imagery and Reanalysis Data

So far we've worked with **point data** — single weather stations measuring temperature at one location. Now we'll explore **spatial data**:

- **Satellite imagery**: Real-time views of the atmosphere from GOES-19
- **Reanalysis data**: Gridded temperature fields covering the entire US

This is like going from listening to one bird at a time to hearing the entire forest at once.

## Step 1: Install & Import Libraries

In [ ]:
!apt-get install -y libgeos-dev libproj-dev -qq
!pip install cartopy xarray netCDF4 -q

import requests
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import Image, display, HTML

## Step 2: GOES-19 Satellite Imagery

The **GOES-19** satellite orbits 35,786 km above the equator, capturing images of the Western Hemisphere every 5 minutes. NOAA provides animated loops of recent imagery.

**GEOCOLOR**: A true-color-like composite that shows clouds, land, and ocean as your eye would see them from space.

In [ ]:
def show_goes(sector="se", product="GEOCOLOR", satellite="GOES19", size=600):
    """Display the latest GOES satellite loop for a given sector and product."""
    url = f"https://cdn.star.nesdis.noaa.gov/{satellite}/ABI/SECTOR/{sector}/{product}/{satellite}-{sector.upper()}-{product}-{size}x{size}.gif"
    print(f"Fetching: {product} for {sector.upper()} sector")
    # Check if the image URL is reachable before trying to display
    try:
        head = requests.head(url, timeout=10)
        if head.status_code != 200:
            print(f"Image not available (HTTP {head.status_code}). The satellite product may be temporarily offline.")
            return
    except requests.exceptions.RequestException:
        print("Could not reach GOES server. Check your internet connection.")
        return
    display(Image(url=url))

# Latest GEOCOLOR for Southeast US
show_goes("se", "GEOCOLOR")

## Step 3: Multiple Satellite Products

Different satellite channels reveal different atmospheric features:

| Product | What It Shows |
|---------|--------------|
| **GEOCOLOR** | True-color clouds, land, ocean |
| **AirMass** | Atmospheric composition (jet stream, dry/moist air) |
| **Sandwich** | Visible imagery draped over infrared (cloud tops + texture) |
| **DayNightCloudMicroCombo** | Cloud particle size and phase |

In [ ]:
products = {
    "GEOCOLOR": "True-color view",
    "AirMass": "Atmospheric composition",
    "Sandwich": "Visible + IR combo",
    "DayNightCloudMicroCombo": "Cloud microphysics",
}

for product, desc in products.items():
    print(f"\n=== {product}: {desc} ===")
    show_goes("eus", product)

## Step 4: NCEP/NCAR Reanalysis Data

**Reanalysis** combines weather observations with atmospheric models to create complete, gridded datasets covering the entire globe. Think of it as "filling in the gaps" between weather stations.

We'll load monthly surface air temperature from NOAA's Physical Sciences Laboratory.

In [ ]:
# Load the NCEP/NCAR reanalysis dataset (this fetches from NOAA)
url = "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.derived/surface/air.mon.mean.nc"
print("Loading NCEP/NCAR Reanalysis data...")
ds = None

# Try OPeNDAP first (fast, streams only the data you need)
try:
    ds = xr.open_dataset(url)
    print("Loaded via OPeNDAP")
except Exception as e:
    print(f"OPeNDAP failed: {e}")

# Fallback: download the full file
if ds is None:
    import urllib.request, os
    local_file = "air.mon.mean.nc"
    if not os.path.exists(local_file):
        print("Downloading full file (~120 MB, may take 1-2 minutes)...")
        urllib.request.urlretrieve(url, local_file)
        print("Download complete.")
    else:
        print("Using previously downloaded file.")
    ds = xr.open_dataset(local_file)

print(f"\nVariables: {list(ds.data_vars)}")
print(f"Dimensions: {dict(ds.dims)}")
print(f"Time range: {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}")
print(f"Lat range: {float(ds.lat.min())} to {float(ds.lat.max())}")
print(f"Lon range: {float(ds.lon.min())} to {float(ds.lon.max())}")

## Step 5: Cartopy Map — Temperature Contours

Extract July 2010 over the Eastern US and plot it on a proper map projection.

In [ ]:
# Extract Eastern US, July 2010
# Note: lon is 0-360 in this dataset, so -100°W = 260°E, -65°W = 295°E
temp = ds["air"].sel(
    time="2010-07-01",
    method="nearest",
    lat=slice(50, 20),
    lon=slice(260, 295)
)

fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-100, -65, 20, 50])

ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linestyle=":")

# Convert lon from 0-360 to -180-180 for plotting
lons = np.where(temp.lon.values > 180, temp.lon.values - 360, temp.lon.values)
c = ax.contourf(lons, temp.lat.values, temp.values,
                transform=ccrs.PlateCarree(),
                cmap="coolwarm", levels=20)

plt.colorbar(c, orientation="vertical", label="Surface Air Temp (°C)", shrink=0.7)
plt.title("NCEP/NCAR Reanalysis: Surface Air Temperature (July 2010)")
plt.show()

## Step 6: Compare Seasons

January vs. July — see how dramatically the temperature field changes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7),
                         subplot_kw={"projection": ccrs.PlateCarree()})

for ax, month, label in zip(axes, ["2010-01-01", "2010-07-01"], ["January 2010", "July 2010"]):
    temp = ds["air"].sel(time=month, method="nearest", lat=slice(50, 20), lon=slice(260, 295))
    lons = np.where(temp.lon.values > 180, temp.lon.values - 360, temp.lon.values)

    ax.set_extent([-100, -65, 20, 50])
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, linestyle=":")

    c = ax.contourf(lons, temp.lat.values, temp.values,
                    transform=ccrs.PlateCarree(),
                    cmap="coolwarm", levels=np.linspace(-15, 35, 25))
    ax.set_title(label, fontsize=14)

plt.colorbar(c, ax=axes, orientation="horizontal", label="Temperature (°C)", shrink=0.6, pad=0.08)
plt.suptitle("Seasonal Contrast: Eastern US Surface Air Temperature", fontsize=16)
plt.show()

## Step 7: Animated Monthly Maps

Watch the temperature field evolve month by month through 2010.

In [ ]:
subset = ds["air"].sel(
    time=slice("2010-01-01", "2010-12-01"),
    lat=slice(50, 20),
    lon=slice(260, 295)
)
lons = np.where(subset.lon.values > 180, subset.lon.values - 360, subset.lon.values)
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent([-100, -65, 20, 50])
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.STATES, linestyle=":")
ax.add_feature(cfeature.BORDERS)

c = ax.contourf(lons, subset.lat.values, subset.isel(time=0).values,
                transform=ccrs.PlateCarree(), cmap="coolwarm",
                levels=np.linspace(-15, 35, 25))
title = ax.set_title("January 2010")
cb = fig.colorbar(c, orientation="vertical", shrink=0.7, label="Temperature (°C)")

def update(frame):
    ax.clear()
    ax.set_extent([-100, -65, 20, 50])
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.STATES, linestyle=":")
    ax.add_feature(cfeature.BORDERS)
    c = ax.contourf(lons, subset.lat.values, subset.isel(time=frame).values,
                    transform=ccrs.PlateCarree(), cmap="coolwarm",
                    levels=np.linspace(-15, 35, 25))
    ax.set_title(f"NCEP/NCAR Reanalysis: {month_labels[frame]} 2010", fontsize=14)
    return c

ani = animation.FuncAnimation(fig, update, frames=12, interval=800, repeat=True)
plt.close(fig)
HTML(ani.to_jshtml())

## Step 8: Explore Your Own Region

Change the variables below to look at any region and time period you want!

In [ ]:
# ====== CHANGE THESE VALUES ======
TARGET_LAT = (50, 20)           # (north, south)
TARGET_LON = (-100, -65)        # (west, east) — use negative for °W, positive for °E
TARGET_TIME = "2015-07-01"      # Any year-month available in the dataset
# =================================

# Auto-convert negative longitudes to 0-360 system used by the dataset
def to_360(lon):
    return lon % 360

lon_west = to_360(TARGET_LON[0])
lon_east = to_360(TARGET_LON[1])

temp = ds["air"].sel(
    time=TARGET_TIME,
    method="nearest",
    lat=slice(TARGET_LAT[0], TARGET_LAT[1]),
    lon=slice(lon_west, lon_east)
)
lons = np.where(temp.lon.values > 180, temp.lon.values - 360, temp.lon.values)

fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([lons.min(), lons.max(), float(temp.lat.min()), float(temp.lat.max())])
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linestyle=":")

c = ax.contourf(lons, temp.lat.values, temp.values,
                transform=ccrs.PlateCarree(), cmap="coolwarm", levels=20)
plt.colorbar(c, label="Temperature (°C)", shrink=0.7)
plt.title(f"Surface Air Temperature: {TARGET_TIME}")
plt.show()

## Station Data vs. Gridded Data

| Feature | Station Data | Gridded Reanalysis |
|---------|-------------|-------------------|
| **Coverage** | Specific points only | Everywhere on the grid |
| **Accuracy** | Direct measurement | Model estimate |
| **Resolution** | Exact location | ~2.5° grid (~250 km) |
| **Best for** | Local trends, long records | Spatial patterns, remote areas |

Both are valuable and complementary. Stations give us ground truth; reanalysis fills the gaps.

## What's Next

In **Notebook 5**, you'll choose your own weather station, fetch its data, and run the full analysis pipeline — from visualization to machine learning clustering.